# D4 関手忠実性 (Functor Faithfulness) 検証実験 — v2 (36動詞)

## Force is Oblivion / Hegemonikón Research (Peira)

**目的**: 外部プロンプトの 90%+ が CCL 式に忠実に変換可能かを 13B モデルで検証

**手法**: 180 プロンプト × 座標ベース決定木 → CCL 変換 → CR/CN/VC 評価

**体系**: HGK v5.0+ — 36動詞 (S/I/A 三値 Flow × 6族 × 2極)

**モデル**: codellama/CodeLlama-13b-Instruct-hf (4bit QLoRA inference)

**GPU**: L4 (24GB) — 13B 4bit ≈ 7GB VRAM

### v1 → v2 変更点
- Flow 座標を I/A 二値 → S/I/A 三値に拡張
- 動詞数を 24 → 36 に増強 (S極12動詞追加)
- コーパスを 120 → 180 に増強 (知覚系プロンプト 60 追加)

### 評価指標
| メトリクス | 成功条件 | 失敗条件 |
|:-----------|:---------|:---------|
| CR (変換率) | >= 90% | < 70% |
| CN (否定正解率) | >= 0.8 | < 0.5 |
| VC (動詞カバー率) | >= 75% (27/36) | < 50% |

In [ ]:
# Cell 1: 環境
!nvidia-smi
!pip install -q transformers bitsandbytes accelerate

In [ ]:
# Cell 2: データアップロード
from google.colab import files
import json

print('corpus_all.jsonl (180 prompts) をアップロードしてください')
uploaded = files.upload()

corpus = []
with open('corpus_all.jsonl') as f:
    for line in f:
        corpus.append(json.loads(line))
print(f'\nCorpus: {len(corpus)} prompts')
for cat in ['analysis','creative','judgment','metacognitive','composite',
            'perception','negative','negative_gray']:
    n = sum(1 for p in corpus if p.get('category') == cat)
    if n: print(f'  {cat}: {n}')

In [ ]:
# Cell 3: モデルロード (CodeLlama-13b-Instruct, 4bit)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'codellama/CodeLlama-13b-Instruct-hf'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb,
    device_map='auto', torch_dtype=torch.bfloat16,
)
print(f'Model: {MODEL_NAME}')
print(f'GPU mem: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
# Cell 4: 変換プロンプトテンプレート (36動詞版)
# HGK v5.0+ — Flow 三値化 (S/I/A)

SYSTEM_PROMPT = """あなたは CCL (Cognitive Control Language) の変換者です。
プロンプトを CCL 式に変換してください。直感ではなく決定木に従ってください。

## Flow 座標 (3値)
S = 知覚 (環境からの入力を受け取る。推論も行為もしない)
I = 推論 (世界モデルを内部的に更新する)
A = 行為 (環境に働きかける)

## 6修飾座標 × 3 Flow = 36動詞

### Telos族 (Value: E=認識目的 / P=実用目的)
S極: /the(S×E 観照する) /ant(S×P 検知する)
I極: /noe(I×E 理解する) /bou(I×P 意志する)
A極: /zet(A×E 探求する) /ene(A×P 実行する)

### Methodos族 (Function: Explore / Exploit)
S極: /ere(S×Ex 探知する) /agn(S×Exp 参照する)
I極: /ske(I×Ex 発散する) /sag(I×Exp 収束する)
A極: /pei(A×Ex 実験する) /tek(A×Exp 適用する)

### Krisis族 (Precision: C=確信 / U=留保)
S極: /sap(S×C 精読する) /ski(S×U 走査する)
I極: /kat(I×C 確定する) /epo(I×U 留保する)
A極: /pai(A×C 決断する) /dok(A×U 打診する)

### Diastasis族 (Scale: Mi=微視 / Ma=巨視)
S極: /prs(S×Mi 注視する) /per(S×Ma 一覧する)
I極: /lys(I×Mi 詳細分析する) /ops(I×Ma 俯瞰する)
A極: /akr(A×Mi 精密操作する) /arh(A×Ma 全体展開する)

### Orexis族 (Valence: +=肯定 / -=否定)
S極: /apo(S×+ 傾聴する) /exe(S×- 吟味する)
I極: /beb(I×+ 肯定する) /ele(I×- 批判する)
A極: /kop(A×+ 推進する) /dio(A×- 是正する)

### Chronos族 (Temporality: Pa=過去 / Fu=未来)
S極: /his(S×Pa 回顧する) /prg(S×Fu 予感する)
I極: /hyp(I×Pa 想起する) /prm(I×Fu 予見する)
A極: /ath(A×Pa 省みる) /par(A×Fu 仕掛ける)

## 決定木
Step 1 — Flow: 環境からの入力収集(S) or 世界モデル更新(I) or 環境への働きかけ(A) or 複数(合成) or 変換不適?
  S の判定基準: ツール駆動の知覚のみ。ファイル読み取り、検索、ログ確認、データ収集、入力の受容的観察。推論や行為を伴わない。
  I の判定基準: 内部的な推論・分析・計画・判断。外部操作を伴わない。
  A の判定基準: 環境への出力。コード実行、ファイル書き込み、コマンド実行、外部への働きかけ。
Step 2 — 主要座標: Value/Function/Precision/Scale/Valence/Temporality のどれが支配的?
Step 2b — 副次座標: あり→(a)修飾 /verb[Coord:極] (b)並行 * (c)順次 ~ / なし / 判断不能
Step 3 — 極の判定: 1動詞に確定
Step 4 — 合成: 複数操作あれば ~, *, >> で接続

## 変換不適の基準
認知操作を含まないプロンプト (情報検索のみ, 純粋計算, 挨拶, 物理的行動) は「変換不適」と回答。
"""

def make_prompt(user_prompt):
    return f"""[INST] <<SYS>>
{SYSTEM_PROMPT}
<</SYS>>

以下のプロンプトを CCL に変換してください。

プロンプト: {user_prompt}

出力形式:
Step 1: {{S/I/A/合成/変換不適}} — 根拠: {{1行}}
Step 2: {{座標名}} — 根拠: {{1行}}
Step 2b: {{副次座標}} — 表現: {{(a)/(b)/(c)/なし/判断不能}}
Step 3: {{極}} — 動詞: {{/略号}}
Step 4: CCL: {{式}}
[/INST]"""

print(f'Template length: {len(make_prompt("test"))} chars')

In [ ]:
# Cell 5: 全プロンプトを変換
import time, re, gc

# OOM 保護: N回ごとに GPU キャッシュを解放
CACHE_CLEAR_INTERVAL = 20

def generate(prompt, max_new_tokens=512):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.3, top_p=0.9,
        )
    result = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    # 中間テンソルを即解放
    del inputs, out
    return result

def parse_ccl(response):
    """応答から CCL 式と変換不適判定を抽出。"""
    r = {'raw': response, 'ccl': None, 'inconvertible': False, 'verb': None, 'flow': None}
    # 変換不適
    if '変換不適' in response:
        r['inconvertible'] = True
        return r
    # Flow 抽出
    m = re.search(r'Step 1:\s*(S|I|A|合成)', response)
    if m:
        r['flow'] = m.group(1)
    # CCL 式抽出
    m = re.search(r'CCL:\s*(.+)', response)
    if m:
        r['ccl'] = m.group(1).strip()
    # 動詞抽出 (Step 3 から。なければ CCL 式から fallback)
    m = re.search(r'動詞:\s*/?(\w+)', response)
    if m:
        r['verb'] = m.group(1)
    elif r['ccl']:
        m2 = re.search(r'/(\w+)', r['ccl'])
        if m2:
            r['verb'] = m2.group(1)
    return r

results = []
t0 = time.time()

for i, p in enumerate(corpus):
    prompt = make_prompt(p['prompt'])
    resp = generate(prompt)
    parsed = parse_ccl(resp)
    parsed['id'] = p['id']
    parsed['category'] = p.get('category', '')
    parsed['prompt'] = p['prompt']
    results.append(parsed)
    
    status = 'INCONV' if parsed['inconvertible'] else (parsed['ccl'] or 'NO_CCL')
    if (i+1) % 10 == 0:
        elapsed = time.time() - t0
        print(f'  [{i+1}/{len(corpus)}] {elapsed:.0f}s — last: {p["id"]} → {status[:40]}')
    
    # OOM 保護: 定期的にキャッシュ解放
    if (i+1) % CACHE_CLEAR_INTERVAL == 0:
        gc.collect()
        torch.cuda.empty_cache()

total_sec = time.time() - t0
print(f'\n変換完了: {len(results)} prompts in {total_sec/60:.1f} min')

In [ ]:
# Cell 6: 評価 (36動詞版)

# --- M1: CR (変換成功率) ---
# 否定テスト以外で CCL が生成されたか
non_neg = [r for r in results if r['category'] not in ('negative', 'negative_gray')]
cr = sum(1 for r in non_neg if r['ccl']) / len(non_neg)

# --- CN: 否定テスト正解率 ---
negatives = [r for r in results if r['category'] == 'negative']
cn = sum(1 for r in negatives if r['inconvertible']) / len(negatives) if negatives else 0

# グレーゾーン
grays = [r for r in results if r['category'] == 'negative_gray']
gray_inconv = sum(1 for r in grays if r['inconvertible']) / len(grays) if grays else 0

# --- カテゴリ別 CR ---
print('='*70)
print('  D4 関手忠実性実験 v2 (36動詞) — 結果')
print('='*70)
print(f'\nM1 変換率 (CR): {cr:.1%} (目標>=90%, 失敗<70%)')
print(f'CN 否定正解率:  {cn:.1%} (目標>=80%)')
print(f'Gray 拒否率:    {gray_inconv:.1%}')

print(f'\nカテゴリ別 CR:')
for cat in ['analysis','creative','judgment','metacognitive','composite','perception']:
    subset = [r for r in results if r['category'] == cat]
    if subset:
        c = sum(1 for r in subset if r['ccl']) / len(subset)
        print(f'  {cat:20s}: {c:.0%} ({sum(1 for r in subset if r["ccl"])}/{len(subset)})')

# --- Flow 分布 ---
print(f'\nFlow 分布:')
flow_counts = {}
for r in results:
    f = r.get('flow') or 'N/A'
    flow_counts[f] = flow_counts.get(f, 0) + 1
for f, c in sorted(flow_counts.items(), key=lambda x: -x[1]):
    if c > 0:
        print(f'  {f}: {c} ({c/len(results):.0%})')

# --- 動詞分布 ---
verbs = {}
for r in results:
    v = r.get('verb', 'none')
    if v:
        verbs[v] = verbs.get(v, 0) + 1
print(f'\n動詞分布 (top 15):')
for v, c in sorted(verbs.items(), key=lambda x: -x[1])[:15]:
    print(f'  /{v}: {c}')

# 36動詞のうち何種類が使われたか
# I/A極 (従来24動詞: 推論12 + 行為12)
ia_verbs = {'noe','bou','zet','ene','ske','sag','pei','tek',
            'kat','epo','pai','dok','lys','ops','akr','arh',
            'beb','ele','kop','dio','hyp','prm','ath','par'}
# S極 (新規12動詞)
s_verbs = {'the','ant','ere','agn','sap','ski',
           'prs','per','apo','exe','his','prg'}
all_verbs = ia_verbs | s_verbs

used = set(verbs.keys()) & all_verbs
used_ia = set(verbs.keys()) & ia_verbs
used_s = set(verbs.keys()) & s_verbs

print(f'\n36動詞中 {len(used)} 種類を使用 (カバー率: {len(used)/36:.0%})')
print(f'  I/A極 (24): {len(used_ia)} 種類')
print(f'  S極   (12): {len(used_s)} 種類')

unused = all_verbs - used
if unused:
    print(f'  未使用: {", ".join(sorted(unused))}')

# --- 族別カバー率 ---
print(f'\n族別カバー率:')
families = {
    'Telos':     {'the','ant','noe','bou','zet','ene'},
    'Methodos':  {'ere','agn','ske','sag','pei','tek'},
    'Krisis':    {'sap','ski','kat','epo','pai','dok'},
    'Diastasis': {'prs','per','lys','ops','akr','arh'},
    'Orexis':    {'apo','exe','beb','ele','kop','dio'},
    'Chronos':   {'his','prg','hyp','prm','ath','par'},
}
for fam, members in families.items():
    fam_used = members & used
    print(f'  {fam:12s}: {len(fam_used)}/6 ({len(fam_used)/6:.0%})')
    if members - used:
        print(f'    未使用: {", ".join(sorted(members - used))}')

# --- D4 判定 ---
print(f'\n{"="*70}')
print('  D4 判定 (36動詞版)')
print(f'{"="*70}')
if cn < 0.8:
    print(f'  ⚠️ CN={cn:.0%} < 80% — 否定テスト不合格。実験全体の信頼性が疑われる')
if cr >= 0.9:
    print(f'  ✅ CR={cr:.0%} >= 90% — D4 (変換率) 支持')
elif cr >= 0.7:
    print(f'  ⚠️ CR={cr:.0%} — D4 弱い支持')
else:
    print(f'  ❌ CR={cr:.0%} < 70% — D4 反証')
vc = len(used) / 36
if vc >= 0.75:
    print(f'  ✅ VC={vc:.0%} >= 75% — 動詞カバー率合格')
else:
    print(f'  ⚠️ VC={vc:.0%} < 75% — 動詞カバー率不足 (S極動詞の出現が少ない可能性)')

In [ ]:
# Cell 7: 結果保存 + ダウンロード
output = {
    'experiment': 'functor_faithfulness_d4_v2_36verbs',
    'model': MODEL_NAME,
    'hgk_version': 'v5.0+ (36 verbs, S/I/A triadic Flow)',
    'n_prompts': len(corpus),
    'metrics': {
        'CR': cr, 'CN': cn, 'gray_inconv': gray_inconv,
        'n_verbs_used': len(used),
        'n_verbs_total': 36,
        'verb_coverage': len(used) / 36,
        'n_ia_verbs_used': len(used_ia),
        'n_s_verbs_used': len(used_s),
    },
    'flow_distribution': {k: v for k, v in flow_counts.items() if v > 0},
    'verb_distribution': dict(sorted(verbs.items(), key=lambda x: -x[1])),
    'family_coverage': {fam: len(members & used) for fam, members in families.items()},
    'conversions': results,
    'gpu': torch.cuda.get_device_name(0),
}

with open('/content/functor_faithfulness_results_v2.json', 'w') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

files.download('/content/functor_faithfulness_results_v2.json')
print('結果ダウンロード')